# Agentic RAG Platform — Local Demo Notebook (No Azure)

This notebook demonstrates all components of the `app/` Agentic RAG platform **locally**, without Docker, FastAPI, or any Azure services.  
It mirrors the architecture of the deployed application but uses **fully local substitutes** so you can run everything on your machine with just an **OpenAI API key**.

## Sequence

1. **Pre-requisites & Environment Setup**
2. **Settings & Configuration**
3. **Local Client Initialisation** (LangChain ChatOpenAI, ChromaDB, SQLite, in-memory dict, local files)
4. **Schemas / Data Models**
5. **Knowledge Base — Ingestion & Search** (ChromaDB with embeddings)
6. **Agent Tools** (Search via ChromaDB, Postgres, Redis, Sandbox)
7. **SRE Agent** (via LangChain / ChatOpenAI)
8. **Engineering Agent** (via LangChain / ChatOpenAI)
9. **Streamlit Chat UI** (replaces FastAPI routes)

---

## Pre-requisites

Before running this notebook make sure you have the following:

### 1. Python ≥ 3.10

### 2. Install dependencies
```bash
pip install openai langchain langchain-openai langchain-text-splitters chromadb pydantic pydantic-settings httpx streamlit
```

### 3. OpenAI API key
Create a `.env` file in the **project root** (`rag-infra/.env`) with at least:
```dotenv
OPENAI_API_KEY=sk-...
OPENAI_MODEL=gpt-4o
LLM_TEMPERATURE=0.2
```
> **Tip:** This notebook uses standard OpenAI (not Azure OpenAI). Get a key at https://platform.openai.com/api-keys  
> ChromaDB uses its built-in `DefaultEmbeddingFunction` (all-MiniLM-L6-v2) for embeddings — no API key needed for that.

## 0. Imports & Path Setup

In [2]:
import sys, os, json, logging, hashlib, asyncio, traceback
from pathlib import Path
from typing import Any, Optional
from enum import Enum

# Ensure the project root is on sys.path so we can reference app/ structure
PROJECT_ROOT = Path.cwd().parent  # assumes notebook is in notebooks/
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Working directory:", os.getcwd())
print("Project root    :", PROJECT_ROOT)

Working directory: c:\ML\AgenticAI\rag-infra
Project root    : c:\ML\AgenticAI\rag-infra


## 1. Settings & Configuration

All configuration lives in `app/core/config.py` (production uses Azure-specific settings).  
For this local demo we define a simplified `Settings` class that reads from `.env` and only requires an **OpenAI API key**.

Below we recreate the `Settings` class following the same pattern as `demo1.ipynb`.

In [3]:
from pydantic_settings import BaseSettings, SettingsConfigDict


class Settings(BaseSettings):
    model_config = SettingsConfigDict(
        env_file=".env",
        env_file_encoding="utf-8",
        env_ignore_empty=True,
        extra="ignore",
    )

    # OpenAI (standard — not Azure)
    openai_api_key: str = ""
    openai_model: str = "gpt-4o"
    llm_temperature: float = 0.2

    # Local data paths
    data_dir: Path = Path("data")
    db_path: Path = Path("data/demo_local.db")
    docs_dir: Path = Path("data/raw-docs")
    chroma_rag_dir: Path = Path("data/chroma_rag")

    # RAG tuning
    rag_chunk_size: int = 800
    rag_chunk_overlap: int = 120
    rag_top_k: int = 5


settings = Settings()

# Ensure directories exist
for d in [settings.data_dir, settings.docs_dir, settings.chroma_rag_dir]:
    (PROJECT_ROOT / d).mkdir(parents=True, exist_ok=True)

print(f"OpenAI API key   : {'***' + settings.openai_api_key[-4:] if settings.openai_api_key else '(not set)'}")
print(f"OpenAI model     : {settings.openai_model}")
print(f"LLM temperature  : {settings.llm_temperature}")
print(f"Docs directory   : {settings.docs_dir}")
print(f"ChromaDB path    : {settings.chroma_rag_dir}")
print(f"RAG chunk size   : {settings.rag_chunk_size}")
print(f"RAG top_k        : {settings.rag_top_k}")

OpenAI API key   : ***U1sA
OpenAI model     : gpt-4o
LLM temperature  : 0.2
Docs directory   : data\raw-docs
ChromaDB path    : data\chroma_rag
RAG chunk size   : 800
RAG top_k        : 5


## 2. Schemas / Data Models

These are the Pydantic models from `app/models/schemas.py`.  
They define the request/response shapes used by the agents and the UI.

In [4]:
from pydantic import BaseModel, Field


class AgentType(str, Enum):
    sre = "sre"
    engineering = "engineering"


class ChatRequest(BaseModel):
    agent: AgentType = AgentType.sre
    session_id: str = Field(..., description="Unique session/conversation ID")
    message: str = Field(..., min_length=1, max_length=4096)


class ChatResponse(BaseModel):
    session_id: str
    agent: AgentType
    answer: str
    sources: list[str] = []
    tool_calls: list[str] = []

class IngestRequest(BaseModel):
    container: str = "raw-docs"
    blob_prefix: str = ""
    force_reindex: bool = False


class IngestResponse(BaseModel):
    status: str
    documents_indexed: int
    errors: list[str] = []


class DocumentItem(BaseModel):
    name: str
    size: int
    last_modified: str
    uri: str


class DocumentsResponse(BaseModel):
    container: str
    documents: list[DocumentItem]


class ServiceStatus(str, Enum):
    ok = "ok"
    degraded = "degraded"
    error = "error"


class HealthResponse(BaseModel):
    status: ServiceStatus
    services: dict[str, Any]


# Quick validation
sample = ChatRequest(agent="sre", session_id="demo-001", message="Why is the API latency high?")
print("Sample ChatRequest:", sample.model_dump_json(indent=2))

Sample ChatRequest: {
  "agent": "sre",
  "session_id": "demo-001",
  "message": "Why is the API latency high?"
}


## 3. Local Client Initialisation

In production the app uses Azure Managed Identity to initialise Key Vault, Blob Storage, AI Search, OpenAI, PostgreSQL, and Redis clients (see `app/core/clients.py`).

For this **local demo** we create:
- A **LangChain ChatOpenAI** LLM client using a standard OpenAI API key (following `demo1.ipynb` pattern)
- A **local in-memory Redis substitute** (dict-based)
- A **local SQLite substitute** for PostgreSQL
- A **local file-based ingestion** substitute for Blob Storage

> No Azure services are required. Only an OpenAI API key is needed for LLM calls.

In [5]:
import sqlite3
from langchain_openai import ChatOpenAI

logger = logging.getLogger("demo")
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")


# ── 3a. LangChain ChatOpenAI LLM (standard OpenAI — no Azure) ───────────────
if not settings.openai_api_key:
    raise ValueError(
        "OPENAI_API_KEY is not set. "
        "Please add it to your .env file in the project root."
    )

llm = ChatOpenAI(
    model=settings.openai_model,
    api_key=settings.openai_api_key,
    temperature=settings.llm_temperature,
)
print(f"✅ LangChain ChatOpenAI initialised (model={settings.openai_model})")


# ── 3b. Local Redis substitute (in-memory dict) ─────────────────────────────
class LocalRedis:
    """Drop-in mock for redis — stores conversation history in a dict."""

    def __init__(self):
        self._store: dict[str, str] = {}

    def get(self, key: str) -> Optional[str]:
        return self._store.get(key)

    def setex(self, key: str, ttl: int, value: str):
        self._store[key] = value  # TTL ignored locally

    def ping(self) -> bool:
        return True


redis_client = LocalRedis()
print("✅ Local Redis (in-memory dict) ready")


# ── 3c. Local SQLite substitute for PostgreSQL ──────────────────────────────
DB_PATH = PROJECT_ROOT / settings.db_path
DB_PATH.parent.mkdir(parents=True, exist_ok=True)


def _get_db() -> sqlite3.Connection:
    conn = sqlite3.connect(str(DB_PATH), check_same_thread=False)
    conn.row_factory = sqlite3.Row
    return conn


def _init_local_pg():
    """Create tables that mirror the production PostgreSQL schema."""
    with _get_db() as conn:
        conn.executescript("""
            CREATE TABLE IF NOT EXISTS incidents (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                service TEXT NOT NULL,
                severity TEXT NOT NULL DEFAULT 'medium',
                title TEXT NOT NULL,
                started_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                resolved_at TIMESTAMP,
                summary TEXT
            );
            CREATE TABLE IF NOT EXISTS service_dependencies (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                upstream TEXT NOT NULL,
                downstream TEXT NOT NULL,
                dependency_type TEXT NOT NULL DEFAULT 'http'
            );
            CREATE TABLE IF NOT EXISTS agent_interactions (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                session_id TEXT NOT NULL,
                agent TEXT NOT NULL,
                query TEXT NOT NULL,
                answer TEXT NOT NULL,
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            );
        """)


_init_local_pg()
print(f"✅ Local SQLite DB initialised at {DB_PATH}")


# ── 3d. Local document directory (replaces Azure Blob Storage) ───────────────
LOCAL_DOCS_DIR = PROJECT_ROOT / settings.docs_dir
LOCAL_DOCS_DIR.mkdir(parents=True, exist_ok=True)
print(f"✅ Local docs directory: {LOCAL_DOCS_DIR}")
print("   Place .md / .txt files here for ingestion")

c:\Users\sayan\anaconda3\envs\py312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ LangChain ChatOpenAI initialised (model=gpt-4o)
✅ Local Redis (in-memory dict) ready
✅ Local SQLite DB initialised at c:\ML\AgenticAI\rag-infra\data\demo_local.db
✅ Local docs directory: c:\ML\AgenticAI\rag-infra\data\raw-docs
   Place .md / .txt files here for ingestion


### 3e. Seed sample data

Insert a few sample incidents and service dependencies so the agent tools have something to query.

In [6]:
with _get_db() as conn:
    # Seed incidents (idempotent — skip if rows exist)
    existing = conn.execute("SELECT COUNT(*) FROM incidents").fetchone()[0]
    if existing == 0:
        conn.executemany(
            "INSERT INTO incidents (service, severity, title, summary) VALUES (?, ?, ?, ?)",
            [
                ("payment-service", "high", "Payment gateway 5xx spike",
                 "Intermittent 502s from Stripe webhook handler; root cause was connection pool exhaustion."),
                ("auth-service", "critical", "OAuth token endpoint down",
                 "Expired TLS cert on auth-service caused 100% failures for 12 minutes."),
                ("order-service", "medium", "Slow order confirmation emails",
                 "SQS queue lag reached 45 s due to under-provisioned consumers."),
                ("payment-service", "low", "Minor logging noise in payment-service",
                 "Debug-level logs flooding CloudWatch; log level bumped to INFO."),
            ],
        )
        print("Seeded 4 sample incidents")
    else:
        print(f"Incidents table already has {existing} rows — skipping seed")

    # Seed service dependencies
    existing_deps = conn.execute("SELECT COUNT(*) FROM service_dependencies").fetchone()[0]
    if existing_deps == 0:
        conn.executemany(
            "INSERT INTO service_dependencies (upstream, downstream, dependency_type) VALUES (?, ?, ?)",
            [
                ("api-gateway", "auth-service", "http"),
                ("api-gateway", "order-service", "http"),
                ("order-service", "payment-service", "http"),
                ("order-service", "inventory-service", "grpc"),
                ("payment-service", "stripe-webhook", "http"),
                ("notification-service", "order-service", "async/sqs"),
            ],
        )
        print("Seeded 6 sample service dependencies")
    else:
        print(f"Dependencies table already has {existing_deps} rows — skipping seed")

Incidents table already has 4 rows — skipping seed
Dependencies table already has 6 rows — skipping seed


## 4. Agent Tools

The platform exposes four tool modules that agents can invoke:

| Tool | Production backend | Local substitute |
|---|---|---|
| `search_tool` — hybrid search + embedding | Azure AI Search + Azure OpenAI Embeddings | **ChromaDB** (persistent vector store + default embeddings) |
| `postgres_tool` — incident history, service deps, audit log | PostgreSQL | Local SQLite |
| `redis_tool` — conversation history & cache | Redis | In-memory dict |
| `sandbox_tool` — restricted code execution | Same (pure Python) | Same (pure Python) |

### 4a. Knowledge Base Service (ChromaDB)

In production this uses Azure AI Search with hybrid (keyword + vector) search and Azure OpenAI embeddings.  
Locally we use **ChromaDB** as a persistent vector store with its built-in `DefaultEmbeddingFunction` (all-MiniLM-L6-v2).  
This follows the same pattern as `demo1.ipynb`'s `KnowledgeBaseService`.

In [7]:
import chromadb
from chromadb.utils import embedding_functions
from langchain_text_splitters import RecursiveCharacterTextSplitter


class KnowledgeBaseService:
    """
    ChromaDB-backed knowledge base (replaces Azure AI Search).
    Chunks documents, embeds them with ChromaDB's DefaultEmbeddingFunction
    (all-MiniLM-L6-v2), and stores them in a persistent local vector store.
    Follows the same pattern as demo1.ipynb's KnowledgeBaseService.
    """

    def __init__(self, settings: Settings):
        self._settings = settings
        chroma_path = PROJECT_ROOT / settings.chroma_rag_dir
        chroma_path.mkdir(parents=True, exist_ok=True)
        self._client = chromadb.PersistentClient(path=str(chroma_path))
        self._collection_name = "rag_knowledge_base"
        self._embedding_function = embedding_functions.DefaultEmbeddingFunction()
        self._collection = self._client.get_or_create_collection(
            name=self._collection_name,
            embedding_function=self._embedding_function,
        )
        self._splitter = RecursiveCharacterTextSplitter(
            chunk_size=settings.rag_chunk_size,
            chunk_overlap=settings.rag_chunk_overlap,
        )

    def ingest_directory(self, directory: Path, clear_existing: bool = False) -> dict[str, int]:
        """Chunk and upsert .md / .txt files into ChromaDB."""
        if clear_existing:
            self._client.delete_collection(name=self._collection_name)
            self._collection = self._client.get_or_create_collection(
                name=self._collection_name,
                embedding_function=self._embedding_function,
            )
        source_files = sorted([*directory.glob("*.md"), *directory.glob("*.txt")])
        docs, ids, metadatas = [], [], []
        for file_path in source_files:
            text = file_path.read_text(encoding="utf-8", errors="ignore")
            chunks = self._splitter.split_text(text)
            for index, chunk in enumerate(chunks):
                chunk_hash = hashlib.sha1(chunk.encode("utf-8")).hexdigest()[:10]
                doc_id = f"{file_path.stem}-{index}-{chunk_hash}"
                docs.append(chunk)
                ids.append(doc_id)
                metadatas.append({"source": file_path.name, "chunk_index": index})
        if docs:
            self._collection.upsert(documents=docs, ids=ids, metadatas=metadatas)
        return {
            "files_indexed": len(source_files),
            "chunks_indexed": len(docs),
            "collection_count": self._collection.count(),
        }

    def search(self, query: str, top_k: int | None = None) -> list[dict[str, Any]]:
        """Query ChromaDB for relevant document chunks."""
        if self._collection.count() == 0:
            return []
        results = self._collection.query(
            query_texts=[query],
            n_results=top_k or self._settings.rag_top_k,
            include=["documents", "metadatas", "distances"],
        )
        documents = (results.get("documents") or [[]])[0]
        metadatas = (results.get("metadatas") or [[]])[0]
        distances = (results.get("distances") or [[]])[0]
        combined = []
        for i, document in enumerate(documents):
            metadata = metadatas[i] if i < len(metadatas) else {}
            distance = distances[i] if i < len(distances) else None
            combined.append({
                "content": document,
                "source": metadata.get("source", "unknown"),
                "distance": distance,
            })
        return combined

    @property
    def count(self) -> int:
        return self._collection.count()


# Initialise the knowledge base
kb = KnowledgeBaseService(settings=settings)
print(f"✅ ChromaDB knowledge base initialised at {PROJECT_ROOT / settings.chroma_rag_dir}")
print(f"   Collection: {kb._collection_name} ({kb.count} chunks)")

# Ingest any docs already in the local docs directory
ingest_stats = kb.ingest_directory(LOCAL_DOCS_DIR)
print(f"   Ingest result: {json.dumps(ingest_stats, indent=2)}")

✅ ChromaDB knowledge base initialised at c:\ML\AgenticAI\rag-infra\data\chroma_rag
   Collection: rag_knowledge_base (8 chunks)
   Ingest result: {
  "files_indexed": 4,
  "chunks_indexed": 8,
  "collection_count": 8
}


### 4a-1. Inspect Knowledge Base Data

Peek at the documents and chunks stored in ChromaDB to verify what the agents will search over.

In [7]:
# ── Inspect all data in the ChromaDB knowledge base ──────────────────────────

# 1. Collection overview
print(f"Collection name : {kb._collection_name}")
print(f"Total chunks    : {kb.count}")
print()

# 2. List all indexed sources and their chunk counts
all_data = kb._collection.get(include=["metadatas", "documents"])
source_counts: dict[str, int] = {}
for meta in all_data["metadatas"]:
    src = meta.get("source", "unknown")
    source_counts[src] = source_counts.get(src, 0) + 1

print("Indexed sources:")
for src, cnt in sorted(source_counts.items()):
    print(f"  {src:40s}  {cnt:>4d} chunks")
print()

# 3. Show a sample of stored chunks (first 5)
sample_size = min(5, len(all_data["documents"]))
print(f"Sample chunks (showing {sample_size} of {len(all_data['documents'])}):")
print("-" * 80)
for i in range(sample_size):
    meta = all_data["metadatas"][i]
    doc = all_data["documents"][i]
    print(f"[{i}] source={meta.get('source', '?')}  chunk_index={meta.get('chunk_index', '?')}")
    print(f"    {doc[:200]}{'...' if len(doc) > 200 else ''}")
    print()

Collection name : rag_knowledge_base
Total chunks    : 8

Indexed sources:
  company-policy.md                            2 chunks
  product-faq.md                               3 chunks
  runbook_payment_service.md                   1 chunks
  technical-runbook.md                         2 chunks

Sample chunks (showing 5 of 8):
--------------------------------------------------------------------------------
[0] source=runbook_payment_service.md  chunk_index=0
    # Payment Service Runbook

## Overview
The payment-service processes all payment transactions via Stripe.

## Common Issues
### 5xx Errors
- Check connection pool settings in `config/pool.yaml`.
- Ver...

[1] source=company-policy.md  chunk_index=0
    # Acme Corp — Remote Work Policy

## Overview
This document outlines the remote work policy for all Acme Corp employees effective January 2026.

## Eligibility
- Full-time employees who have completed...

[2] source=company-policy.md  chunk_index=1
    ## Security
- VPN must b

### 4b. Postgres Tool (local SQLite version)

Mirrors `app/agents/tools/postgres_tool.py` — incident history, service dependencies, and audit logging.

In [8]:
def query_incident_history(service_name: str, limit: int = 10) -> list[dict]:
    """Query recent incidents for a given service (local SQLite version)."""
    with _get_db() as conn:
        rows = conn.execute(
            "SELECT id, service, severity, title, started_at, resolved_at, summary "
            "FROM incidents WHERE service = ? ORDER BY started_at DESC LIMIT ?",
            (service_name, limit),
        ).fetchall()
    return [dict(r) for r in rows]


def query_service_dependencies(service_name: str) -> list[dict]:
    """Query service dependency graph (local SQLite version)."""
    with _get_db() as conn:
        rows = conn.execute(
            "SELECT upstream, downstream, dependency_type "
            "FROM service_dependencies WHERE upstream = ? OR downstream = ?",
            (service_name, service_name),
        ).fetchall()
    return [dict(r) for r in rows]


def log_agent_interaction(session_id: str, agent: str, query: str, answer: str):
    """Persist agent interaction for audit (local SQLite version)."""
    with _get_db() as conn:
        conn.execute(
            "INSERT INTO agent_interactions (session_id, agent, query, answer) VALUES (?, ?, ?, ?)",
            (session_id, agent, query, answer),
        )


# Test the tools
print("Incidents for 'payment-service':")
for inc in query_incident_history("payment-service"):
    print(f"  [{inc['severity']}] {inc['title']}")

print("\nDependencies involving 'order-service':")
for dep in query_service_dependencies("order-service"):
    print(f"  {dep['upstream']} → {dep['downstream']} ({dep['dependency_type']})")

Incidents for 'payment-service':
  [high] Payment gateway 5xx spike
  [low] Minor logging noise in payment-service

Dependencies involving 'order-service':
  api-gateway → order-service (http)
  order-service → payment-service (http)
  order-service → inventory-service (grpc)
  notification-service → order-service (async/sqs)


### 4c. Redis Tool (local in-memory version)

Mirrors `app/agents/tools/redis_tool.py` — conversation history and caching.

In [9]:
HISTORY_TTL = 3600  # 1 hour (ignored locally, kept for parity)


def get_conversation_history(session_id: str) -> list[dict]:
    """Retrieve conversation history from local Redis substitute."""
    raw = redis_client.get(f"session:{session_id}:history")
    if not raw:
        return []
    return json.loads(raw)


def append_to_history(session_id: str, role: str, content: str):
    """Append a message to the conversation history (keeps last 20 turns)."""
    history = get_conversation_history(session_id)
    history.append({"role": role, "content": content})
    history = history[-20:]
    redis_client.setex(
        f"session:{session_id}:history",
        HISTORY_TTL,
        json.dumps(history),
    )


def cache_set(key: str, value: str, ttl: int = 300):
    redis_client.setex(key, ttl, value)


def cache_get(key: str) -> Optional[str]:
    return redis_client.get(key)


# Test
append_to_history("demo-001", "user", "Hello, what is the payment-service?")
append_to_history("demo-001", "assistant", "The payment-service handles all payment processing.")
history = get_conversation_history("demo-001")
print(f"Conversation history ({len(history)} messages):")
for msg in history:
    print(f"  [{msg['role']}] {msg['content'][:80]}")

Conversation history (2 messages):
  [user] Hello, what is the payment-service?
  [assistant] The payment-service handles all payment processing.


### 4d. Sandbox Tool

Executes untrusted Python code in a restricted sandbox.  
This is the same implementation as `app/agents/tools/sandbox_tool.py` — no Azure dependency.

In [10]:
import builtins as _builtins_mod

SANDBOX_TIMEOUT = 10  # seconds
ALLOWED_BUILTINS = {
    "print", "len", "range", "enumerate", "zip",
    "list", "dict", "set", "tuple", "str", "int",
    "float", "bool", "type", "isinstance", "min", "max", "sum",
}


def execute_code(code: str) -> dict[str, Any]:
    """
    Execute untrusted Python code in a restricted sandbox.
    No file I/O, no imports, no network — pure computation only.
    (Synchronous version for notebook use; production uses async.)
    """
    restricted_globals = {
        "__builtins__": {k: getattr(_builtins_mod, k) for k in ALLOWED_BUILTINS if hasattr(_builtins_mod, k)},
    }
    output_lines: list[str] = []

    def _capture_print(*args, **kwargs):
        output_lines.append(" ".join(str(a) for a in args))

    restricted_globals["__builtins__"]["print"] = _capture_print

    try:
        local_vars: dict = {}
        exec(compile(code, "<sandbox>", "exec"), restricted_globals, local_vars)
        return {
            "status": "ok",
            "output": "\n".join(output_lines),
            "locals": {k: repr(v) for k, v in local_vars.items()},
        }
    except Exception:
        return {"status": "error", "output": traceback.format_exc(limit=5)}


# Test
result = execute_code("x = sum(range(10))\nprint('Sum:', x)")
print("Sandbox result:", json.dumps(result, indent=2))

Sandbox result: {
  "status": "ok",
  "output": "Sum: 45",
  "locals": {
    "x": "45"
  }
}


## 5. RAG Service — Ingestion & Retrieval

`retrieve_context()` mirrors `app/services/rag_service.py`.  
It searches **ChromaDB** and builds a context string for the LLM.

`ingest_local_docs()` mirrors `app/services/ingestion_service.py`.  
It reads files from `data/raw-docs/` and stores chunks in ChromaDB.

In [11]:
CONTEXT_MAX_CHARS = 6000


def retrieve_context(query: str, top_k: int = 5) -> tuple[str, list[str]]:
    """
    Retrieve relevant chunks from ChromaDB and build a context string.
    Mirrors app/services/rag_service.py but uses ChromaDB instead of Azure AI Search.
    """
    hits = kb.search(query, top_k=top_k)

    context_parts = []
    sources = []
    total_chars = 0

    for hit in hits:
        chunk = f"[{hit['source']}]\n{hit['content']}"
        if total_chars + len(chunk) > CONTEXT_MAX_CHARS:
            break
        context_parts.append(chunk)
        sources.append(hit["source"])
        total_chars += len(chunk)

    context = "\n\n---\n\n".join(context_parts)
    return context, list(set(sources))


def ingest_local_docs(directory: Path | None = None, clear_existing: bool = False) -> dict[str, int]:
    """
    Ingest .md and .txt files from a local directory into ChromaDB.
    Mirrors app/services/ingestion_service.py (which reads from Azure Blob Storage).
    """
    target = directory or LOCAL_DOCS_DIR
    return kb.ingest_directory(target, clear_existing=clear_existing)


# Create a sample document for testing if the docs directory is empty
sample_doc = LOCAL_DOCS_DIR / "runbook_payment_service.md"
if not sample_doc.exists():
    sample_doc.write_text(
        "# Payment Service Runbook\n\n"
        "## Overview\n"
        "The payment-service processes all payment transactions via Stripe.\n\n"
        "## Common Issues\n"
        "### 5xx Errors\n"
        "- Check connection pool settings in `config/pool.yaml`.\n"
        "- Verify Stripe API key is valid and not rate-limited.\n"
        "- Inspect CloudWatch logs for `PoolExhausted` exceptions.\n\n"
        "### High Latency\n"
        "- Check database connection pool utilisation.\n"
        "- Review recent deployments for regression.\n"
        "- Verify downstream Stripe endpoint health at https://status.stripe.com.\n\n"
        "## Escalation\n"
        "If unresolved within 15 minutes, page the payments-oncall rotation.\n",
        encoding="utf-8",
    )
    print("Created sample runbook:", sample_doc.name)
    # Re-ingest with the new sample doc
    ingest_result = ingest_local_docs()
    print("Re-ingest result:", json.dumps(ingest_result, indent=2))

# Test retrieval
context, sources = retrieve_context("payment service 5xx errors")
print(f"\nRetrieved context ({len(context)} chars) from {len(sources)} source(s)")
if context:
    print(context[:500])


Retrieved context (3263 chars) from 4 source(s)
[runbook_payment_service.md]
# Payment Service Runbook

## Overview
The payment-service processes all payment transactions via Stripe.

## Common Issues
### 5xx Errors
- Check connection pool settings in `config/pool.yaml`.
- Verify Stripe API key is valid and not rate-limited.
- Inspect CloudWatch logs for `PoolExhausted` exceptions.

### High Latency
- Check database connection pool utilisation.
- Review recent deployments for regression.
- Verify downstream Stripe endpoint health at https://s


## 6. SRE Agent

The SRE agent (`app/agents/sre_agent.py`) analyses incidents, suggests root cause analysis, and helps with on-call triage.

It combines:
1. Conversation history (local Redis substitute)
2. RAG context (local keyword search)
3. Incident history (local SQLite)
4. Sandbox execution (if code block in message)
5. LLM call via **LangChain ChatOpenAI** (standard OpenAI API)

In [12]:
from langchain_core.messages import HumanMessage, SystemMessage

SRE_SYSTEM_PROMPT = """
You are an expert SRE (Site Reliability Engineer) AI assistant.
Your responsibilities:
- Analyze incidents and alerts based on historical data and runbooks.
- Suggest root cause analysis (RCA) and remediation steps.
- Help with on-call triage, runbook lookup, and postmortem drafting.
- Answer questions about service dependencies and SLOs/SLIs.
- Execute diagnostic code snippets safely when needed.

Always:
- Ground your answers in retrieved context from the knowledge base.
- Cite sources when referencing runbooks or past incidents.
- Be concise, structured (use numbered steps for procedures).
- Never reveal secrets, connection strings, or internal credentials.
"""

# Common English words to skip during service name detection
_STOP_WORDS = {
    "what", "when", "where", "which", "there", "their", "about",
    "would", "could", "should", "have", "been", "that", "this",
    "with", "from", "your", "into", "will", "more", "also",
}


def run_sre_agent(session_id: str, user_message: str) -> dict[str, Any]:
    """
    Local synchronous version of app/agents/sre_agent.py.
    Uses local tools instead of Azure services.
    Calls OpenAI via LangChain ChatOpenAI.
    """
    logger.info("SRE agent: session=%s query='%s'", session_id, user_message[:80])

    # 1) Conversation history from local Redis
    history = []
    try:
        history = get_conversation_history(f"sre:{session_id}")
    except Exception as exc:
        logger.warning("History fetch failed: %s", exc)

    # 2) RAG context from local search index
    context, sources = retrieve_context(user_message, top_k=settings.rag_top_k)

    # 3) Incident lookup from local SQLite
    incidents = []
    tool_calls = []
    words = user_message.lower().split()
    for word in words:
        if len(word) > 4 and word not in _STOP_WORDS:
            rows = query_incident_history(word)
            if rows:
                incidents = rows
                tool_calls.append(f"query_incident_history(service={word})")
                break

    # 4) Sandbox execution (if code block detected)
    sandbox_result = None
    if "```python" in user_message:
        start = user_message.find("```python") + 9
        end = user_message.find("```", start)
        if end > start:
            code_block = user_message[start:end].strip()
            sandbox_result = execute_code(code_block)
            tool_calls.append("execute_code(sandbox)")

    # 5) Build LangChain messages
    messages = [SystemMessage(content=SRE_SYSTEM_PROMPT)]

    if context:
        messages.append(SystemMessage(content=f"Relevant knowledge base context:\n\n{context}"))

    if incidents:
        incident_text = "\n".join(
            f"- [{r['severity']}] {r['title']} at {r['started_at']}: {r.get('summary', '')}"
            for r in incidents
        )
        messages.append(SystemMessage(content=f"Recent incidents:\n{incident_text}"))

    if sandbox_result:
        messages.append(SystemMessage(
            content=f"Sandbox execution result:\nStatus: {sandbox_result['status']}\nOutput:\n{sandbox_result['output']}",
        ))

    # Add conversation history as LangChain messages
    for msg in history[-10:]:
        if msg["role"] == "user":
            messages.append(HumanMessage(content=msg["content"]))
        else:
            from langchain_core.messages import AIMessage
            messages.append(AIMessage(content=msg["content"]))

    messages.append(HumanMessage(content=user_message))

    # 6) Call OpenAI via LangChain ChatOpenAI
    try:
        response = llm.invoke(messages)
        answer = response.content
    except Exception as exc:
        answer = f"[LLM call failed: {exc}]\n\nContext retrieved:\n{context[:500] if context else 'None'}"

    # 7) Persist to local Redis + SQLite
    try:
        append_to_history(f"sre:{session_id}", "user", user_message)
        append_to_history(f"sre:{session_id}", "assistant", answer)
    except Exception as exc:
        logger.warning("History save failed: %s", exc)

    try:
        log_agent_interaction(session_id, "sre", user_message, answer)
    except Exception as exc:
        logger.warning("Interaction log failed: %s", exc)

    return {"answer": answer, "sources": sources, "tool_calls": tool_calls}


print("✅ SRE agent defined")

✅ SRE agent defined


### 6a. Test the SRE Agent

In [13]:
sre_result = run_sre_agent(
    session_id="demo-sre-001",
    user_message="We're seeing 5xx errors on payment-service. What should I check first?",
)

print("=" * 60)
print("SRE AGENT RESPONSE")
print("=" * 60)
print(sre_result["answer"])
print(f"\nSources: {sre_result['sources']}")
print(f"Tool calls: {sre_result['tool_calls']}")

INFO | SRE agent: session=demo-sre-001 query='We're seeing 5xx errors on payment-service. What should I check first?'
INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


SRE AGENT RESPONSE
To address 5xx errors on the payment-service, follow these steps:

1. **Check Connection Pool Settings**: Inspect the configuration in `config/pool.yaml` to ensure that the connection pool settings are correctly configured.

2. **Verify Stripe API Key**: Ensure that the Stripe API key is valid and not rate-limited. This can often be a cause of service disruptions if the key is invalid or has exceeded its usage limits.

3. **Inspect CloudWatch Logs**: Look for `PoolExhausted` exceptions in the CloudWatch logs. This can indicate that the connection pool is exhausted, leading to 5xx errors.

If these steps do not resolve the issue within 15 minutes, escalate by paging the payments-oncall rotation as per the escalation procedure in the runbook. (Source: [runbook_payment_service.md])

Sources: ['technical-runbook.md', 'company-policy.md', 'product-faq.md', 'runbook_payment_service.md']
Tool calls: []


### 6b. SRE Agent — Latency Spike Investigation

In [ ]:
sre_result_2 = run_sre_agent(
    session_id="demo-sre-002",
    user_message="We're observing p99 latency spikes on order-service above 2s. How should I triage this?",
)

print("=" * 60)
print("SRE AGENT RESPONSE — Latency Spike")
print("=" * 60)
print(sre_result_2["answer"])
print(f"\nSources: {sre_result_2['sources']}")
print(f"Tool calls: {sre_result_2['tool_calls']}")

### 6c. SRE Agent — Runbook Lookup with Code Execution

In [ ]:
sre_result_3 = run_sre_agent(
    session_id="demo-sre-003",
    user_message=(
        "Our monitoring shows Redis connection pool exhaustion on notification-service. "
        "Can you pull up the relevant runbook steps and also run this diagnostic?\n\n"
        "```python\nstats = {'active_connections': 48, 'max_pool': 50, 'idle': 2}\n"
        "utilization = stats['active_connections'] / stats['max_pool'] * 100\n"
        "print(f'Pool utilization: {utilization:.1f}%')\n"
        "print('CRITICAL' if utilization > 90 else 'OK')\n```"
    ),
)

print("=" * 60)
print("SRE AGENT RESPONSE — Runbook + Sandbox Execution")
print("=" * 60)
print(sre_result_3["answer"])
print(f"\nSources: {sre_result_3['sources']}")
print(f"Tool calls: {sre_result_3['tool_calls']}")

## 7. Engineering Agent

The Engineering agent (`app/agents/engineering_agent.py`) answers architecture, design, and code-related questions.

It combines:
1. Conversation history (local Redis substitute)
2. RAG context (local keyword search)
3. Service dependency lookup (local SQLite)
4. Sandbox execution (if code block in message)
5. LLM call via **LangChain ChatOpenAI** (standard OpenAI API)

In [ ]:
from langchain_core.messages import AIMessage

ENGINEERING_SYSTEM_PROMPT = """
You are an expert Software Engineering AI assistant embedded in an engineering platform.
Your responsibilities:
- Answer architecture, design, and code-related questions.
- Review code snippets and suggest improvements.
- Explain service dependencies and integration patterns.
- Help with debugging, performance analysis, and best practices.
- Execute safe code snippets in a sandboxed environment.

Always:
- Ground answers in retrieved internal documentation.
- Cite sources (ADRs, RFCs, wiki pages) when relevant.
- Be precise and use concrete examples.
- Prefer idiomatic, production-ready code suggestions.
- Never output secrets, credentials, or connection strings.
"""


def run_engineering_agent(session_id: str, user_message: str) -> dict[str, Any]:
    """
    Local synchronous version of app/agents/engineering_agent.py.
    Uses local tools instead of Azure services.
    Calls OpenAI via LangChain ChatOpenAI.
    """
    logger.info("Engineering agent: session=%s query='%s'", session_id, user_message[:80])

    # 1) Conversation history
    history = []
    try:
        history = get_conversation_history(f"eng:{session_id}")
    except Exception as exc:
        logger.warning("History fetch failed: %s", exc)

    # 2) RAG context
    context, sources = retrieve_context(user_message, top_k=settings.rag_top_k)

    # 3) Service dependency lookup
    dependencies = []
    tool_calls = []
    words = user_message.lower().split()
    for word in words:
        if len(word) > 4 and word not in _STOP_WORDS:
            rows = query_service_dependencies(word)
            if rows:
                dependencies = rows
                tool_calls.append(f"query_service_dependencies(service={word})")
                break

    # 4) Sandbox execution (if code block detected)
    sandbox_result = None
    if "```python" in user_message:
        start = user_message.find("```python") + 9
        end = user_message.find("```", start)
        if end > start:
            code_block = user_message[start:end].strip()
            sandbox_result = execute_code(code_block)
            tool_calls.append("execute_code(sandbox)")

    # 5) Build LangChain messages
    messages = [SystemMessage(content=ENGINEERING_SYSTEM_PROMPT)]

    if context:
        messages.append(SystemMessage(content=f"Relevant internal documentation:\n\n{context}"))

    if dependencies:
        dep_text = "\n".join(
            f"- {r['upstream']} → {r['downstream']} ({r['dependency_type']})"
            for r in dependencies
        )
        messages.append(SystemMessage(content=f"Service dependencies:\n{dep_text}"))

    if sandbox_result:
        messages.append(SystemMessage(
            content=f"Sandbox execution result:\nStatus: {sandbox_result['status']}\nOutput:\n{sandbox_result['output']}",
        ))

    # Add conversation history as LangChain messages
    for msg in history[-10:]:
        if msg["role"] == "user":
            messages.append(HumanMessage(content=msg["content"]))
        else:
            messages.append(AIMessage(content=msg["content"]))

    messages.append(HumanMessage(content=user_message))

    # 6) Call OpenAI via LangChain ChatOpenAI
    try:
        response = llm.invoke(messages)
        answer = response.content
    except Exception as exc:
        answer = f"[LLM call failed: {exc}]\n\nContext retrieved:\n{context[:500] if context else 'None'}"

    # 7) Persist history + audit log
    try:
        append_to_history(f"eng:{session_id}", "user", user_message)
        append_to_history(f"eng:{session_id}", "assistant", answer)
    except Exception as exc:
        logger.warning("History save failed: %s", exc)

    try:
        log_agent_interaction(session_id, "engineering", user_message, answer)
    except Exception as exc:
        logger.warning("Interaction log failed: %s", exc)

    return {"answer": answer, "sources": sources, "tool_calls": tool_calls}


print("✅ Engineering agent defined")

### 7a. Test the Engineering Agent

In [ ]:
eng_result = run_engineering_agent(
    session_id="demo-eng-001",
    user_message="What are the dependencies of order-service and how does it connect to payment-service?",
)

print("=" * 60)
print("ENGINEERING AGENT RESPONSE")
print("=" * 60)
print(eng_result["answer"])
print(f"\nSources: {eng_result['sources']}")
print(f"Tool calls: {eng_result['tool_calls']}")

### 7b. Engineering Agent — Architecture Review with Code Snippet

In [ ]:
eng_result_2 = run_engineering_agent(
    session_id="demo-eng-002",
    user_message=(
        "Review the retry logic below for our payment-service client and suggest improvements:\n\n"
        "```python\nimport time\n\ndef call_payment(payload, retries=3):\n"
        "    for i in range(retries):\n"
        "        try:\n"
        "            resp = http_client.post('/pay', json=payload)\n"
        "            resp.raise_for_status()\n"
        "            return resp.json()\n"
        "        except Exception:\n"
        "            time.sleep(1)\n"
        "    raise RuntimeError('payment call failed after retries')\n\n"
        "print('Retry logic defined')\n```"
    ),
)

print("=" * 60)
print("ENGINEERING AGENT RESPONSE — Code Review + Sandbox")
print("=" * 60)
print(eng_result_2["answer"])
print(f"\nSources: {eng_result_2['sources']}")
print(f"Tool calls: {eng_result_2['tool_calls']}")

## 8. Health Check

Mirrors `app/api/routes/health.py` — verifies connectivity to all backend services.

In [ ]:
def check_health() -> dict[str, Any]:
    """
    Local health check — mirrors app/api/routes/health.py.
    Checks local Redis, SQLite, LLM client, and ChromaDB.
    """
    services = {}
    overall = "ok"

    # Redis
    try:
        redis_client.ping()
        services["redis"] = "ok (local in-memory)"
    except Exception as exc:
        services["redis"] = f"error: {exc}"
        overall = "degraded"

    # PostgreSQL (SQLite)
    try:
        with _get_db() as conn:
            conn.execute("SELECT 1").fetchone()
        services["postgres"] = "ok (local SQLite)"
    except Exception as exc:
        services["postgres"] = f"error: {exc}"
        overall = "degraded"

    # OpenAI via LangChain — verify with a lightweight call
    try:
        llm.invoke([HumanMessage(content="ping")])
        services["openai"] = f"ok (model={settings.openai_model})"
    except Exception as exc:
        services["openai"] = f"degraded: {exc}"
        overall = "degraded"

    # ChromaDB
    try:
        count = kb.count
        services["chromadb"] = f"ok ({count} chunks indexed)"
    except Exception as exc:
        services["chromadb"] = f"error: {exc}"
        overall = "degraded"

    return {"status": overall, "services": services}


health = check_health()
print("Health Check:")
print(json.dumps(health, indent=2))

## 9. Streamlit Chat UI

Instead of FastAPI routes (`app/api/routes/chat.py`, `ingest.py`, `documents.py`, `health.py`), we provide a **Streamlit** app that exposes the same functionality through a browser UI.

The cell below writes a `streamlit_app.py` file to disk and provides instructions to run it.

### Features
- **Agent selector** — choose between SRE and Engineering agents
- **Chat interface** — multi-turn conversation with history
- **Document ingestion** — upload `.md` / `.txt` files into the local search index
- **Health dashboard** — shows status of all local services

In [ ]:
streamlit_app_code = r'''
"""
Streamlit Chat UI for Agentic RAG Platform (No Azure)
=====================================================
Replaces the FastAPI routes with a simple browser-based interface.
Uses standard OpenAI via LangChain ChatOpenAI + ChromaDB for RAG — no Azure dependency.

Run with:
    streamlit run streamlit_app_noAzure.py
"""
import streamlit as st
import sys, os, json, hashlib, logging, sqlite3, traceback, uuid
from pathlib import Path
from typing import Any, Optional
from enum import Enum

# ── Ensure project root is importable ─────────────────────────────────────────
PROJECT_ROOT = Path(__file__).resolve().parent.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pydantic_settings import BaseSettings, SettingsConfigDict
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
import chromadb
from chromadb.utils import embedding_functions
from langchain_text_splitters import RecursiveCharacterTextSplitter

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("streamlit_app")

# ══════════════════════════════════════════════════════════════════════════════
# Settings
# ══════════════════════════════════════════════════════════════════════════════
class Settings(BaseSettings):
    model_config = SettingsConfigDict(env_file=".env", env_file_encoding="utf-8", env_ignore_empty=True, extra="ignore")
    openai_api_key: str = ""
    openai_model: str = "gpt-4o"
    llm_temperature: float = 0.2
    data_dir: Path = Path("data")
    db_path: Path = Path("data/demo_local.db")
    docs_dir: Path = Path("data/raw-docs")
    chroma_rag_dir: Path = Path("data/chroma_rag")
    rag_chunk_size: int = 800
    rag_chunk_overlap: int = 120
    rag_top_k: int = 5

settings = Settings()

# ══════════════════════════════════════════════════════════════════════════════
# LangChain ChatOpenAI LLM (required)
# ══════════════════════════════════════════════════════════════════════════════
if not settings.openai_api_key:
    st.error(
        "❌ OPENAI_API_KEY must be set in .env file. "
        "Please add it and restart the app."
    )
    st.stop()

llm = ChatOpenAI(
    model=settings.openai_model,
    api_key=settings.openai_api_key,
    temperature=settings.llm_temperature,
)

# ══════════════════════════════════════════════════════════════════════════════
# Local clients
# ══════════════════════════════════════════════════════════════════════════════
class LocalRedis:
    def __init__(self):
        self._store: dict[str, str] = {}
    def get(self, key: str) -> Optional[str]:
        return self._store.get(key)
    def setex(self, key: str, ttl: int, value: str):
        self._store[key] = value
    def ping(self) -> bool:
        return True

redis_client = LocalRedis()

DB_PATH = PROJECT_ROOT / settings.db_path
DB_PATH.parent.mkdir(parents=True, exist_ok=True)

def _get_db() -> sqlite3.Connection:
    conn = sqlite3.connect(str(DB_PATH), check_same_thread=False)
    conn.row_factory = sqlite3.Row
    return conn

LOCAL_DOCS_DIR = PROJECT_ROOT / settings.docs_dir
LOCAL_DOCS_DIR.mkdir(parents=True, exist_ok=True)

# ══════════════════════════════════════════════════════════════════════════════
# ChromaDB Knowledge Base
# ══════════════════════════════════════════════════════════════════════════════
class KnowledgeBaseService:
    def __init__(self, settings):
        chroma_path = PROJECT_ROOT / settings.chroma_rag_dir
        chroma_path.mkdir(parents=True, exist_ok=True)
        self._client = chromadb.PersistentClient(path=str(chroma_path))
        self._collection_name = "rag_knowledge_base"
        self._ef = embedding_functions.DefaultEmbeddingFunction()
        self._collection = self._client.get_or_create_collection(
            name=self._collection_name, embedding_function=self._ef,
        )
        self._splitter = RecursiveCharacterTextSplitter(
            chunk_size=settings.rag_chunk_size, chunk_overlap=settings.rag_chunk_overlap,
        )
        self._settings = settings

    def ingest_directory(self, directory, clear_existing=False):
        if clear_existing:
            self._client.delete_collection(name=self._collection_name)
            self._collection = self._client.get_or_create_collection(
                name=self._collection_name, embedding_function=self._ef,
            )
        files = sorted([*directory.glob("*.md"), *directory.glob("*.txt")])
        docs, ids, metas = [], [], []
        for fp in files:
            text = fp.read_text(encoding="utf-8", errors="ignore")
            for i, chunk in enumerate(self._splitter.split_text(text)):
                h = hashlib.sha1(chunk.encode()).hexdigest()[:10]
                docs.append(chunk); ids.append(f"{fp.stem}-{i}-{h}")
                metas.append({"source": fp.name, "chunk_index": i})
        if docs:
            self._collection.upsert(documents=docs, ids=ids, metadatas=metas)
        return {"files_indexed": len(files), "chunks_indexed": len(docs), "collection_count": self._collection.count()}

    def search(self, query, top_k=None):
        if self._collection.count() == 0:
            return []
        r = self._collection.query(query_texts=[query], n_results=top_k or self._settings.rag_top_k,
                                    include=["documents", "metadatas", "distances"])
        docs = (r.get("documents") or [[]])[0]
        metas = (r.get("metadatas") or [[]])[0]
        dists = (r.get("distances") or [[]])[0]
        return [{"content": docs[i], "source": (metas[i] or {}).get("source", "unknown"),
                 "distance": dists[i] if i < len(dists) else None} for i in range(len(docs))]

    @property
    def count(self):
        return self._collection.count()

kb = KnowledgeBaseService(settings)

def retrieve_context(query, top_k=5):
    hits = kb.search(query, top_k=top_k)
    parts, srcs, total = [], [], 0
    for h in hits:
        chunk = f"[{h['source']}]\n{h['content']}"
        if total + len(chunk) > 6000:
            break
        parts.append(chunk); srcs.append(h["source"]); total += len(chunk)
    return "\n\n---\n\n".join(parts), list(set(srcs))

# ══════════════════════════════════════════════════════════════════════════════
# DB Tools
# ══════════════════════════════════════════════════════════════════════════════
def query_incident_history(service_name, limit=10):
    with _get_db() as conn:
        return [dict(r) for r in conn.execute(
            "SELECT * FROM incidents WHERE service = ? ORDER BY started_at DESC LIMIT ?",
            (service_name, limit)).fetchall()]

def query_service_dependencies(service_name):
    with _get_db() as conn:
        return [dict(r) for r in conn.execute(
            "SELECT upstream, downstream, dependency_type FROM service_dependencies WHERE upstream = ? OR downstream = ?",
            (service_name, service_name)).fetchall()]

def log_agent_interaction(session_id, agent, query, answer):
    with _get_db() as conn:
        conn.execute("INSERT INTO agent_interactions (session_id, agent, query, answer) VALUES (?, ?, ?, ?)",
                     (session_id, agent, query, answer))

def get_conversation_history(session_id):
    raw = redis_client.get(f"session:{session_id}:history")
    return json.loads(raw) if raw else []

def append_to_history(session_id, role, content):
    h = get_conversation_history(session_id)
    h.append({"role": role, "content": content})
    redis_client.setex(f"session:{session_id}:history", 3600, json.dumps(h[-20:]))

import builtins as _bm
_AB = {"print","len","range","enumerate","zip","list","dict","set","tuple","str","int","float","bool","type","isinstance","min","max","sum"}
def execute_code(code):
    rg = {"__builtins__": {k: getattr(_bm, k) for k in _AB if hasattr(_bm, k)}}
    ol = []
    rg["__builtins__"]["print"] = lambda *a, **kw: ol.append(" ".join(str(x) for x in a))
    try:
        lv = {}
        exec(compile(code, "<sandbox>", "exec"), rg, lv)
        return {"status": "ok", "output": "\n".join(ol)}
    except Exception:
        return {"status": "error", "output": traceback.format_exc(limit=5)}

_STOP_WORDS = {"what","when","where","which","there","their","about","would","could","should","have","been","that","this","with","from","your","into","will","more","also"}

# ══════════════════════════════════════════════════════════════════════════════
# Agent runners (using LangChain ChatOpenAI)
# ══════════════════════════════════════════════════════════════════════════════
SRE_PROMPT = "You are an expert SRE AI assistant. Analyze incidents, suggest RCA and remediation. Ground answers in retrieved context. Be concise."
ENG_PROMPT = "You are an expert Software Engineering AI assistant. Answer architecture and code questions. Ground answers in retrieved docs. Be precise."

def _run_agent(session_id, user_message, agent_type):
    prefix = "sre" if agent_type == "sre" else "eng"
    system_prompt = SRE_PROMPT if agent_type == "sre" else ENG_PROMPT
    history = get_conversation_history(f"{prefix}:{session_id}")
    context, sources = retrieve_context(user_message, top_k=settings.rag_top_k)
    tool_calls = []

    db_context = ""
    for word in user_message.lower().split():
        if len(word) > 4 and word not in _STOP_WORDS:
            if agent_type == "sre":
                rows = query_incident_history(word)
                if rows:
                    db_context = "Recent incidents:\n" + "\n".join(f"- [{r['severity']}] {r['title']}: {r.get('summary','')}" for r in rows)
                    tool_calls.append(f"query_incident_history(service={word})")
                    break
            else:
                rows = query_service_dependencies(word)
                if rows:
                    db_context = "Service dependencies:\n" + "\n".join(f"- {r['upstream']} → {r['downstream']} ({r['dependency_type']})" for r in rows)
                    tool_calls.append(f"query_service_dependencies(service={word})")
                    break

    sandbox_result = None
    if "```python" in user_message:
        s = user_message.find("```python") + 9
        e = user_message.find("```", s)
        if e > s:
            sandbox_result = execute_code(user_message[s:e].strip())
            tool_calls.append("execute_code(sandbox)")

    msgs = [SystemMessage(content=system_prompt)]
    if context:
        msgs.append(SystemMessage(content=f"Retrieved context:\n\n{context}"))
    if db_context:
        msgs.append(SystemMessage(content=db_context))
    if sandbox_result:
        msgs.append(SystemMessage(content=f"Sandbox result:\n{sandbox_result['output']}"))
    for msg in history[-10:]:
        if msg["role"] == "user":
            msgs.append(HumanMessage(content=msg["content"]))
        else:
            msgs.append(AIMessage(content=msg["content"]))
    msgs.append(HumanMessage(content=user_message))

    try:
        resp = llm.invoke(msgs)
        answer = resp.content
    except Exception as exc:
        answer = f"[LLM error: {exc}]\n\nContext:\n{context[:500]}" if context else f"[LLM error: {exc}]"

    append_to_history(f"{prefix}:{session_id}", "user", user_message)
    append_to_history(f"{prefix}:{session_id}", "assistant", answer)
    try:
        log_agent_interaction(session_id, agent_type, user_message, answer)
    except Exception:
        pass
    return {"answer": answer, "sources": sources, "tool_calls": tool_calls}

# ══════════════════════════════════════════════════════════════════════════════
# Startup: ingest local docs into ChromaDB
# ══════════════════════════════════════════════════════════════════════════════
kb.ingest_directory(LOCAL_DOCS_DIR)

# ══════════════════════════════════════════════════════════════════════════════
# Streamlit UI
# ══════════════════════════════════════════════════════════════════════════════
st.set_page_config(page_title="Agentic RAG Chat", page_icon="🤖", layout="wide")
st.title("🤖 Agentic RAG Platform — Local Demo")

tab_chat, tab_ingest, tab_health = st.tabs(["💬 Chat", "📄 Ingest Documents", "🩺 Health"])

with tab_chat:
    col1, col2 = st.columns([1, 3])
    with col1:
        agent_type = st.radio("Agent", ["sre", "engineering"], index=0)
        session_id = st.text_input("Session ID", value=str(uuid.uuid4())[:8])

    with col2:
        if "messages" not in st.session_state:
            st.session_state.messages = []

        for msg in st.session_state.messages:
            with st.chat_message(msg["role"]):
                st.markdown(msg["content"])

        if prompt := st.chat_input("Ask the agent..."):
            st.session_state.messages.append({"role": "user", "content": prompt})
            with st.chat_message("user"):
                st.markdown(prompt)

            with st.chat_message("assistant"):
                with st.spinner("Thinking..."):
                    result = _run_agent(session_id, prompt, agent_type)
                st.markdown(result["answer"])
                if result["sources"]:
                    st.caption(f"📚 Sources: {', '.join(result['sources'])}")
                if result["tool_calls"]:
                    st.caption(f"🔧 Tools used: {', '.join(result['tool_calls'])}")

            st.session_state.messages.append({"role": "assistant", "content": result["answer"]})

with tab_ingest:
    st.subheader("Upload documents to the knowledge base")
    uploaded = st.file_uploader("Choose .md or .txt files", type=["md", "txt"], accept_multiple_files=True)
    if uploaded and st.button("Ingest"):
        for f in uploaded:
            dest = LOCAL_DOCS_DIR / f.name
            dest.write_bytes(f.getvalue())
        stats = kb.ingest_directory(LOCAL_DOCS_DIR)
        st.success(f"Ingested {stats['chunks_indexed']} chunks from {stats['files_indexed']} file(s)")

with tab_health:
    st.subheader("Service Health")
    svc = {}
    svc["Redis"] = "✅ ok (local in-memory)"
    try:
        with _get_db() as c:
            c.execute("SELECT 1")
        svc["PostgreSQL"] = "✅ ok (local SQLite)"
    except Exception as e:
        svc["PostgreSQL"] = f"❌ {e}"
    svc["OpenAI"] = f"✅ configured (model={settings.openai_model})"
    svc["ChromaDB"] = f"✅ {kb.count} chunks indexed"
    for name, status in svc.items():
        st.markdown(f"**{name}**: {status}")
'''

# Write the Streamlit app to disk
streamlit_path = PROJECT_ROOT / "notebooks" / "streamlit_app_noAzure.py"
streamlit_path.write_text(streamlit_app_code.strip(), encoding="utf-8")
print(f"✅ Streamlit app written to: {streamlit_path}")
print()
print("To run the Streamlit app, open a terminal and execute:")
print(f"    cd {PROJECT_ROOT / 'notebooks'}")
print("    streamlit run streamlit_app_noAzure.py")

## 10. Auth Flow (Reference)

In the production app, authentication is handled by `app/core/auth.py` using **Azure Entra ID (AAD)** JWT tokens.

The Streamlit demo does **not** include authentication. Below is the reference implementation for context.

> To add auth to Streamlit, you could use [streamlit-authenticator](https://github.com/mkhorasani/Streamlit-Authenticator) or any OAuth2 provider.

In [ ]:
# ── Auth reference (from app/core/auth.py) ────────────────────────────────────
# This section is informational only — auth is NOT enforced in the local demo.
#
# Production auth flow (uses Azure Entra ID):
#   1. Client sends a Bearer token in the Authorization header.
#   2. The token is a JWT issued by the identity provider.
#   3. The server validates the JWT signature and claims.
#   4. Role-based access checks the "roles" claim.
#
# To add auth to the Streamlit app:
#   - Use streamlit-authenticator for simple user/password auth.
#   - Or use any OAuth2/OIDC provider with the requests_oauthlib library.
#   - Or protect the Streamlit app behind a reverse proxy with auth.

print("ℹ️  Auth section is reference-only — no authentication is enforced in this local demo.")

## 11. Document Storage (Reference)

In production, document ingestion reads from **Azure Blob Storage** (`app/services/ingestion_service.py`).

Locally this is replaced by the `data/raw-docs/` directory — no cloud storage needed.

In [ ]:
# ── Document Storage reference ─────────────────────────────────────────────────
# Production uses Azure Blob Storage; locally we use data/raw-docs/ directory.
#
# Production ingestion flow:
#   1. Client calls POST /ingest with container name and optional blob prefix.
#   2. The service iterates over blobs in the cloud storage container.
#   3. Each .txt / .md / .pdf file is downloaded and chunked.
#   4. Chunks are embedded and uploaded to the search index.
#
# Local equivalent:
#   - Place .md or .txt files in data/raw-docs/
#   - Run ingest_local_docs() or use the Streamlit "Ingest Documents" tab
#   - Documents are chunked and stored in the in-memory _local_search_index

# List locally available documents
local_docs = sorted(LOCAL_DOCS_DIR.glob("*"))
print(f"📁 Documents in {LOCAL_DOCS_DIR}:")
for doc in local_docs:
    size = doc.stat().st_size if doc.is_file() else 0
    print(f"  {doc.name:40s}  {size:>8,d} bytes")

## 12. Vector Store (Reference)

In production, the RAG service uses Azure AI Search with vector + semantic search and Azure OpenAI embeddings.

Locally we use **ChromaDB** as a persistent vector store with its built-in `DefaultEmbeddingFunction` (all-MiniLM-L6-v2). Below is a comparison of the two setups.

In [ ]:
# ── Vector store reference (from app/services/rag_service.py) ──────────────────
#
# Production search index schema (Azure AI Search):
#   Fields:
#     - id             : String (key)
#     - content        : Searchable String
#     - title          : Searchable String (filterable)
#     - source         : Simple String (filterable)
#     - content_vector : Collection(Single), 3072 dimensions, HNSW profile
#
#   Hybrid search (keyword + vector):
#     - query_text   → keyword search on content + title
#     - query_vector → nearest-neighbor search on content_vector
#     - Results merged by reciprocal rank fusion
#
# Local substitute — ChromaDB (persistent vector store):
#     - PersistentClient  → data stored at data/chroma_rag/
#     - DefaultEmbeddingFunction (all-MiniLM-L6-v2, ~384-d)
#     - collection.query(query_texts=[...]) → automatic embedding + ANN search
#     - No API key required for embeddings

print("ℹ️  Vector store section is reference-only.")
print(f"   Local demo uses ChromaDB with {kb.count} chunks indexed.")

## 13. Audit Log — Review Agent Interactions

All agent interactions are persisted in the local SQLite database. This is useful for debugging and reviewing agent behaviour.

In [ ]:
with _get_db() as conn:
    rows = conn.execute(
        "SELECT session_id, agent, query, created_at FROM agent_interactions ORDER BY created_at DESC LIMIT 10"
    ).fetchall()

print(f"Recent agent interactions ({len(rows)}):")
print("-" * 90)
for r in rows:
    row = dict(r)
    print(f"  [{row['created_at']}] {row['agent']:12s} session={row['session_id']}")
    print(f"    Q: {row['query'][:80]}")
    print()

---

## Summary

This notebook demonstrated every layer of the **Agentic RAG Platform** running locally with **zero Azure dependencies**:

| Section | Production Component | Local Substitute |
|---|---|---|
| Settings | `app/core/config.py` (Azure settings) | Simplified Pydantic Settings with `OPENAI_API_KEY` |
| Schemas | `app/models/schemas.py` | Same Pydantic models |
| LLM Client | Azure OpenAI (Managed Identity) | **LangChain ChatOpenAI** (standard OpenAI API key) |
| Search / Vector Store | Azure AI Search (hybrid) + Azure OpenAI Embeddings | **ChromaDB** (persistent vector store + default embeddings) |
| Postgres Tool | Azure PostgreSQL | Local SQLite |
| Redis Tool | Azure Cache for Redis | In-memory dict |
| Sandbox Tool | Restricted `exec()` | Same |
| RAG Service | Azure AI Search + Azure OpenAI Embeddings | **ChromaDB** vector search via `KnowledgeBaseService` |
| SRE Agent | `app/agents/sre_agent.py` (Azure OpenAI) | LangChain `llm.invoke()` |
| Engineering Agent | `app/agents/engineering_agent.py` (Azure OpenAI) | LangChain `llm.invoke()` |
| API / UI | FastAPI + Azure Entra ID auth | **Streamlit** (no auth) |
| Health Check | `app/api/routes/health.py` | Local health function |
| Ingestion | Azure Blob → Azure AI Search | Local files → **ChromaDB** vector store |

### Next Steps
- Add `.md` / `.txt` runbooks to `data/raw-docs/` to enrich the knowledge base
- Set `OPENAI_API_KEY` in `.env` for real LLM responses
- Run `streamlit run notebooks/streamlit_app_noAzure.py` for the interactive chat UI